In [ ]:
import os
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle
import numpy as np
import scipy.io
import tensorflow as tf
import joblib
import numpy as np
import pandas as pd
from tabulate import tabulate

In [ ]:
from tensorflow.keras.layers import Conv1D, Dense, Flatten, Dropout, InputLayer, MaxPooling1D, BatchNormalization
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.optimizers import Adam
from larq.layers import QuantConv1D, QuantDense
import larq as lq

In [ ]:
# Check if GPU is Available
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    # Print information about each GPU
    for gpu in gpus:
        print("GPU:", gpu.name)
else:
    print("No GPU devices found.")

## Dataset Preparation

In [ ]:
def dataset(classes, val=True, window_length=5000, seed=42):
    seed = random.randint(0, 1000) if seed is None else seed
    if classes is None:
        raise ValueError("Classes cannot be None.")
    
    class_paths = []
    for classNum in classes:
        class_paths.append('./DatasetPDT/' + classNum + '/avt/')
    print(class_paths)

    ts_len = 65536

    # Load data
    ts_list = []
    labels_list = []
    for class_path in class_paths:
        print(class_path)
        for path in os.listdir(class_path):
            if path.endswith(".mat"):
                mat = scipy.io.loadmat(class_path + path)
                data = mat['data']
                data = data.T

                for i in range(len(data)):
                    # Check if length of timeserie is less than ts_len
                    if len(data[i]) < ts_len:
                        np_data = np.array(data[i])
                        last_value = np_data[-1]
                        # Concatenate the last value until ts_len
                        additional_values = np.full(ts_len - len(np_data), last_value)
                        np_data = np.concatenate((np_data, additional_values))
                        # Add to data
                        ts_list.append(np.array(np_data))
                    else:
                        # Add directly to data
                        ts_list.append(np.array(data[i]))
                    # Add label
                    labels_list.append(class_paths.index(class_path))

    # Combine dataset and labels for shuffling
    combined_data = list(zip(ts_list, labels_list))
    # Shuffle the combined data
    shuffled_data = shuffle(combined_data, random_state=seed)
    combined_data = None
    # Split the shuffled data back into dataset and labels
    shuffled_dataset, shuffled_labels = zip(*shuffled_data)
    shuffled_dataset = np.array(shuffled_dataset)
    shuffled_labels = np.array(shuffled_labels)

    scaler = StandardScaler()
    # Fit the scaler on the training data and transform both training and validation data
    scaled_dataset = scaler.fit_transform(shuffled_dataset.reshape(-1, ts_len)).reshape(shuffled_dataset.shape)

    combined_data = list(zip(scaled_dataset, shuffled_labels))
    train_data, test_data, val_data = [], [], []

    for class_label in range(len(classes)):
        class_data, class_labels = zip(*[(data, label) for data, label in combined_data if label == class_label])
        train, test, ytrain, ytest = train_test_split(class_data, class_labels, test_size=1/5, random_state=seed)
        train, valid, ytrain, yval = train_test_split(train, ytrain, test_size=1/8, random_state=seed)
        train_data.extend(zip(train, ytrain))
        test_data.extend(zip(test, ytest))
        val_data.extend(zip(valid, yval))

    # Unzip and convert the data into numpy arrays
    X_train_pre, y_train_pre = map(np.array, zip(*train_data))
    X_test_pre, y_test_pre = map(np.array, zip(*test_data))
    X_val_pre, y_val_pre = map(np.array, zip(*val_data))

    X_train_pre = np.array(X_train_pre)
    X_test_pre = np.array(X_test_pre)
    X_val_pre = np.array(X_val_pre)
    y_train_pre = np.array(y_train_pre)
    y_test_pre = np.array(y_test_pre)
    y_val_pre = np.array(y_val_pre)

    def create_windows(data_array, label_array, window_length):
        X, y = [], []
        for i in range(len(data_array)):
            for start in range(0, len(data_array[i]) - window_length + 1, window_length):
                end = start + window_length
                X.append(data_array[i][start:end])
                y.append(label_array[i])
        return X, y

    X_train, y_train = create_windows(X_train_pre, y_train_pre, window_length)
    X_val, y_val = create_windows(X_val_pre, y_val_pre, window_length)
    X_test, y_test = create_windows(X_test_pre, y_test_pre, window_length)
    
    width = len(X_train[0])
 
    # Assuming X_train, X_valid, and X_test are 2D arrays with shape (number_of_samples, sequence_length)
    X_train = np.array([np.array(x) for x in X_train])
    X_val = np.array([np.array(x) for x in X_val])
    X_test = np.array([np.array(x) for x in X_test])
    
    # Reshape the input data to have a third dimension
    X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1) 
    X_val = X_val.reshape(X_val.shape[0], X_val.shape[1], 1)     
    X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

    print(X_train.shape, X_val.shape, X_test.shape)   
    
    X_train = tf.convert_to_tensor(X_train, dtype=tf.float32)
    y_train = tf.convert_to_tensor(y_train, dtype=tf.float32)
    X_val = tf.convert_to_tensor(X_val, dtype=tf.float32)
    y_val = tf.convert_to_tensor(y_val, dtype=tf.float32)
    X_test = tf.convert_to_tensor(X_test, dtype=tf.float32)
    y_test = tf.convert_to_tensor(y_test, dtype=tf.float32)
    
    return X_train, X_val, X_test, y_train, y_val, y_test, width

2025-03-05 17:59:53.235969: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-03-05 17:59:53.511629: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-05 17:59:53.588823: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-03-05 17:59:53.588839: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not ha

In [2]:
classes = ['01','03','04','05', '06','07','09','10','11', '12','13','14','15','16','17']
X_train, X_val, X_test, y_train, y_val, y_test, width= dataset(classes, val=True, window_length=65536)

['./DatasetPDT/01/avt/', './DatasetPDT/03/avt/', './DatasetPDT/04/avt/', './DatasetPDT/05/avt/', './DatasetPDT/06/avt/', './DatasetPDT/07/avt/', './DatasetPDT/09/avt/', './DatasetPDT/10/avt/', './DatasetPDT/11/avt/', './DatasetPDT/12/avt/', './DatasetPDT/13/avt/', './DatasetPDT/14/avt/', './DatasetPDT/15/avt/', './DatasetPDT/16/avt/', './DatasetPDT/17/avt/']
./DatasetPDT/01/avt/
./DatasetPDT/03/avt/
./DatasetPDT/04/avt/
./DatasetPDT/05/avt/
./DatasetPDT/06/avt/
./DatasetPDT/07/avt/
./DatasetPDT/09/avt/
./DatasetPDT/10/avt/
./DatasetPDT/11/avt/
./DatasetPDT/12/avt/
./DatasetPDT/13/avt/
./DatasetPDT/14/avt/
./DatasetPDT/15/avt/
./DatasetPDT/16/avt/
./DatasetPDT/17/avt/
(3021, 65536, 1) (432, 65536, 1) (877, 65536, 1)


2025-03-05 18:00:25.202379: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:966] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-03-05 18:00:25.202591: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2025-03-05 18:00:25.202823: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublas.so.11'; dlerror: libcublas.so.11: cannot open shared object file: No such file or directory
2025-03-05 18:00:25.203020: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcublasLt.so.11'; dlerror: libcublasLt.so.11: cannot open shared object file: No such file or directory
2025-03-05 18:00:25.203054: W tensorflow/stream_executor/platform/default/dso_loader.cc:6